In [12]:
import cv2
import numpy as np
import os


class RoadDefectDetector:

    def __init__(self):

        self.clahe = cv2.createCLAHE(
            clipLimit=3.0,
            tileGridSize=(8, 8)
        )

    def get_auto_roi(self, frame):

        h, w = frame.shape[:2]

        mask = np.zeros((h, w), dtype=np.uint8)

        points = np.array([[
            [int(w * 0.02), h],
            [int(w * 0.08), int(h * 0.18)],
            [int(w * 0.92), int(h * 0.18)],
            [int(w * 0.98), h]
        ]], dtype=np.int32)

        cv2.fillPoly(mask, points, 255)

        return mask

    def enhance(self, frame):

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        blur = cv2.GaussianBlur(gray, (5, 5), 0)

        enhanced = self.clahe.apply(blur)

        return enhanced

    def process_frame(self, frame):

        roi_mask = self.get_auto_roi(frame)

        enhanced = self.enhance(frame)

        road_only = cv2.bitwise_and(
            enhanced,
            enhanced,
            mask=roi_mask
        )

        result = frame.copy()

        detected_types = set()

        binary = cv2.adaptiveThreshold(
            road_only,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            15,
            5
        )

        threshold_output = binary.copy()

        kernel = np.ones((3, 3), np.uint8)

        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        morphology_output = binary.copy()

        contours, _ = cv2.findContours(
            binary,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for cnt in contours:

            area = cv2.contourArea(cnt)

            if area < 100:
                continue

            x, y, w, h = cv2.boundingRect(cnt)

            aspect_ratio = float(w) / h

            if area > 1500:

                hull = cv2.convexHull(cnt)

                cv2.drawContours(result, [hull], -1, (255, 0, 0), 3)

                detected_types.add("Pothole")

            elif aspect_ratio > 4.0 and w > 80:

                cv2.drawContours(result, [cnt], -1, (0, 0, 255), 2)

                detected_types.add("Transverse Crack")

            elif (aspect_ratio < 0.25 and h > 80):

                cv2.drawContours(result, [cnt], -1, (0, 255, 255), 2)

                detected_types.add("Edge Crack")

            elif 500 < area < 4000:

                cv2.drawContours(result, [cnt], -1, (0, 255, 0), 1)

                detected_types.add("Ravelling")

        for i, text in enumerate(detected_types):

            cv2.putText(
                result,
                f"DETECTED: {text}",
                (20, 40 + (i * 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        return (
            result,
            enhanced,
            threshold_output,
            morphology_output
        )


# =========================================================
# MAIN EXECUTION
# =========================================================

video_input = r"D:\image processing project\enhanced.mp4"
video_output = r"D:\image processing project\detected_output.mp4"

base_output = r"D:\segmentation\outputs"

enhanced_folder = os.path.join(base_output, "enhanced_frames")
threshold_folder = os.path.join(base_output, "threshold_masks")
morphology_folder = os.path.join(base_output, "morphological_refinement")
final_folder = os.path.join(base_output, "final_outputs")
comparison_folder = os.path.join(base_output, "comparisons")

os.makedirs(enhanced_folder, exist_ok=True)
os.makedirs(threshold_folder, exist_ok=True)
os.makedirs(morphology_folder, exist_ok=True)
os.makedirs(final_folder, exist_ok=True)
os.makedirs(comparison_folder, exist_ok=True)

detector = RoadDefectDetector()

cap = cv2.VideoCapture(video_input)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    video_output,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

print("Processing Video... Press 'q' to stop.")

frame_count = 0

# =========================================================
# FIX: SAFE FRAME READING LOOP (NO FRAME LOSS)
# =========================================================

while True:

    ret, frame = cap.read()

    if not ret:

        retry = 0

        while retry < 5:
            ret, frame = cap.read()

            if ret:
                break

            retry += 1

        if not ret:
            print("End of video or unreadable frame reached.")
            break

    processed, enhanced_frame, threshold_mask, morphology_mask = detector.process_frame(frame)

    # SAVE OUTPUTS
    cv2.imwrite(os.path.join(enhanced_folder, f"enhanced_{frame_count:05d}.png"), enhanced_frame)
    cv2.imwrite(os.path.join(threshold_folder, f"threshold_{frame_count:05d}.png"), threshold_mask)
    cv2.imwrite(os.path.join(morphology_folder, f"morphology_{frame_count:05d}.png"), morphology_mask)
    cv2.imwrite(os.path.join(final_folder, f"final_{frame_count:05d}.png"), processed)

    # COMPARISON
    enhanced_bgr = cv2.cvtColor(enhanced_frame, cv2.COLOR_GRAY2BGR)
    threshold_bgr = cv2.cvtColor(threshold_mask, cv2.COLOR_GRAY2BGR)
    morphology_bgr = cv2.cvtColor(morphology_mask, cv2.COLOR_GRAY2BGR)

    comparison = np.hstack([
        frame,
        enhanced_bgr,
        threshold_bgr,
        morphology_bgr,
        processed
    ])

    cv2.imwrite(os.path.join(comparison_folder, f"comparison_{frame_count:05d}.png"), comparison)

    out.write(processed)

    cv2.imshow("Automated Road Defect Detection", processed)

    frame_count += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done!")

Processing Video... Press 'q' to stop.
End of video or unreadable frame reached.
Done!
